# 🧠 Alzheimer's MRI Classifier — Training Notebook

This notebook walks through:
1. Dataset exploration
2. VGG16 fine-tuning
3. ResNet50 fine-tuning
4. Ensemble evaluation
5. Saving models for Flask inference

**Dataset**: [Kaggle Alzheimer's MRI Disease](https://www.kaggle.com/datasets/tourist55/alzheimers-dataset-4-class-of-images)
Download and extract to `backend/data/` before running.

In [ ]:
import os, sys
# Navigate to backend root so imports work
os.chdir(os.path.join(os.getcwd(), '..', '..'))
sys.path.insert(0, os.getcwd())
print('Working dir:', os.getcwd())

In [ ]:
# ── Dataset exploration ────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import os

DATA_DIR  = 'data'
CLASSES   = ['MildDemented', 'ModerateDemented', 'NonDemented', 'VeryMildDemented']
COLORS    = ['#e74c3c', '#f39c12', '#2ecc71', '#3498db']

# Count images per class
counts = {}
for split in ['train', 'test']:
    counts[split] = {}
    for cls in CLASSES:
        p = os.path.join(DATA_DIR, split, cls)
        counts[split][cls] = len(os.listdir(p)) if os.path.isdir(p) else 0

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, split in zip(axes, ['train', 'test']):
    vals = [counts[split][c] for c in CLASSES]
    bars = ax.bar(CLASSES, vals, color=COLORS, edgecolor='white', linewidth=1.5)
    ax.set_title(f'{split.capitalize()} Set Distribution', fontsize=14, fontweight='bold')
    ax.set_ylabel('Image count')
    ax.tick_params(axis='x', rotation=20)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                str(val), ha='center', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()
print('Total train:', sum(counts['train'].values()))
print('Total test: ', sum(counts['test'].values()))

In [ ]:
# ── Sample images from each class ─────────────────────────────────────────────
fig, axes = plt.subplots(4, 4, figsize=(12, 12))
for row, cls in enumerate(CLASSES):
    cls_dir = os.path.join(DATA_DIR, 'train', cls)
    imgs    = os.listdir(cls_dir)[:4]
    for col, img_name in enumerate(imgs):
        img = Image.open(os.path.join(cls_dir, img_name)).convert('RGB')
        axes[row][col].imshow(img, cmap='gray')
        axes[row][col].axis('off')
        if col == 0:
            axes[row][col].set_title(cls.replace('Demented','\nDemented'), fontsize=9)
plt.suptitle('Sample MRI Images by Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Train VGG16 ────────────────────────────────────────────────────────────────
from ml.training.build_vgg16 import _build as train_vgg16

os.makedirs('models', exist_ok=True)
vgg_model, h1, h2 = train_vgg16(
    data_dir    = 'data',
    epochs      = 20,
    output_path = 'models/vgg16_alzheimer.h5',
    batch_size  = 32
)

In [ ]:
# ── Plot VGG16 training curves ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for hist, label in [(h1, 'Stage 1'), (h2, 'Stage 2')]:
    axes[0].plot(hist.history['accuracy'],     label=f'{label} Train')
    axes[0].plot(hist.history['val_accuracy'], label=f'{label} Val', linestyle='--')
    axes[1].plot(hist.history['loss'],         label=f'{label} Train')
    axes[1].plot(hist.history['val_loss'],     label=f'{label} Val',  linestyle='--')

axes[0].set_title('VGG16 Accuracy');   axes[0].legend(); axes[0].set_xlabel('Epoch')
axes[1].set_title('VGG16 Loss');       axes[1].legend(); axes[1].set_xlabel('Epoch')
plt.tight_layout(); plt.show()

In [ ]:
# ── Train ResNet50 ─────────────────────────────────────────────────────────────
from ml.training.build_resnet50 import _build as train_resnet

res_model = train_resnet(
    data_dir    = 'data',
    epochs      = 20,
    output_path = 'models/resnet50_alzheimer.h5',
    batch_size  = 32
)

In [ ]:
# ── Ensemble evaluation ────────────────────────────────────────────────────────
from ml.training.evaluate import evaluate

evaluate(
    data_dir    = 'data',
    vgg16_path  = 'models/vgg16_alzheimer.h5',
    resnet50_path='models/resnet50_alzheimer.h5'
)

In [ ]:
# ── Test single image inference ────────────────────────────────────────────────
from services.mri_service import predict_mri

# Pick any test image
test_img = os.path.join(DATA_DIR, 'test', 'MildDemented',
               os.listdir(os.path.join(DATA_DIR, 'test', 'MildDemented'))[0])

with open(test_img, 'rb') as f:
    result = predict_mri(f.read(), filename=os.path.basename(test_img))

print('\n=== Prediction Result ===')
for k, v in result.items():
    print(f'  {k:30s}: {v}')